In [1]:
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python311.zip')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/lib-dynload')
sys.path.append('/home/amunif/.local/lib/python3.11/site-packages')
sys.path.append('/group/pmc021/amunif/env/pytorch/lib/python3.11/site-packages')

In [2]:
import os
import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score, dcg_score, classification_report

import xgboost as xgb

In [3]:
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/'
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/09_Learning to Rank/'

In [4]:
def load_data(path):
    gene_pl = pd.read_parquet(path)
    return gene_pl

In [5]:
def reformat_data(df, columns):
    processed_arrays = []

    for col in columns:
        stacked = np.vstack(df[col].values)
        processed_arrays.append(stacked)

    X = np.hstack(processed_arrays)
    return X

# Load Dataset

In [6]:
# Load dataset
gene_pl = load_data(os.path.join(DATASET_DIR, 'dataset', 'gene_w_label_value_1.parquet'))
gene_pl.head(5)

,gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
0,XLOC_000001,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.000000,0
1,XLOC_000003,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.000000,0
2,XLOC_000006,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.088845,0
3,XLOC_000007,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,4.047430,1
4,XLOC_000008,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",6,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,26.793400,1


In [7]:
markers = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']
X = reformat_data(gene_pl, markers)

In [8]:
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [9]:
y = gene_pl['label'].values

In [10]:
gene_ids = gene_pl['gene_id'].values
gene_ids

array(['XLOC_000001', 'XLOC_000003', 'XLOC_000006', ..., 'XLOC_030014',
       'XLOC_030017', 'XLOC_030018'], dtype=object)

In [11]:
value_1 = gene_pl['value_1'].values

In [12]:
y[:20]

array([0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1],
      dtype=int32)

In [13]:
seed = 1994
rng = np.random.default_rng(seed)
n_query_groups = 1 # Single query group
qid = rng.integers(0, n_query_groups, size=X.shape[0])

# Modeling

In [14]:
ranker = xgb.XGBRanker(
            tree_method="hist", 
            lambdarank_num_pair_per_sample=8, 
            objective="rank:pairwise", 
            lambdarank_pair_method="topk"
        )

In [15]:
X.shape

(22154, 20000)

In [16]:
# Create dataframe for ranking
df = pd.DataFrame(X, columns=[str(i) for i in range(X.shape[1])])
df["qid"] = qid

In [17]:
df

,0,1,2,3,4,5,6,7,8,9,...,19991,19992,19993,19994,19995,19996,19997,19998,19999,qid
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22149,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
22150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
22151,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
22152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [18]:
ranker.fit(df, y)

XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=None, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, gamma=None, grow_policy=None,
          importance_type=None, interaction_constraints=None,
          lambdarank_num_pair_per_sample=8, lambdarank_pair_method='topk',
          learning_rate=None, max_bin=None, max_cat_threshold=None,
          max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
          max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=None,
          n_jobs=None, ...)

In [19]:
scores = ranker.predict(X)

In [20]:
df_full = df

In [21]:
df_full['gene_id'] = gene_ids

In [22]:
df_full['value_1'] = value_1

In [23]:
df_full['y'] = y

In [24]:
df_full['scores'] = scores

In [25]:
df_full

,0,1,2,3,4,5,6,7,8,9,...,19995,19996,19997,19998,19999,qid,gene_id,value_1,y,scores
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000001,0.000000,0,-6.970554
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000003,0.000000,0,-6.970554
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000006,0.088845,0,-6.970554
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000007,4.047430,1,-6.970554
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000008,26.793400,1,-5.069316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22149,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_030009,0.000000,0,-6.970554
22150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_030012,0.000000,0,-6.970554
22151,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_030014,0.000000,0,-6.970554
22152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_030017,0.000000,0,-6.970554


In [26]:
df_full.sort_values(by=['scores'], ascending=[False])

,0,1,2,3,4,5,6,7,8,9,...,19995,19996,19997,19998,19999,qid,gene_id,value_1,y,scores
5814,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_007268,131.084000,1,5.207706
5994,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_007485,18.302500,1,5.207706
19866,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_026833,17.541200,1,5.174291
20253,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_027299,26.474600,1,5.174291
1155,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_001315,21.046700,1,5.005527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1865,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_002096,0.021255,0,-7.192238
13479,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_017257,0.183057,0,-7.192238
4956,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_006168,7.119170,1,-7.192238
9811,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_012585,4.412670,1,-7.192238


In [27]:
df_full['scores'].describe()

count    22154.000000
mean        -5.684216
std          1.598334
min         -7.192238
25%         -6.970554
50%         -6.324584
75%         -4.800021
max          5.207706
Name: scores, dtype: float64

In [28]:
df_full.sort_values(by=['value_1'], ascending=[False])

,0,1,2,3,4,5,6,7,8,9,...,19995,19996,19997,19998,19999,qid,gene_id,value_1,y,scores
4492,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_005626,12870.90,1,-6.970554
15660,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_020078,9933.06,1,-6.970554
10487,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_013407,8005.12,1,-0.523247
421,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_000470,5475.90,1,-6.970554
1859,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_002089,5094.75,1,-6.970554
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12549,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_015856,0.00,0,-4.649570
12546,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_015853,0.00,0,-6.970554
12538,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_015845,0.00,0,-6.467713
12519,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0,XLOC_015825,0.00,0,-6.970554


In [29]:
df_full[["gene_id", "value_1", "y", "scores"]].to_csv("HepG2_ranking_binary.csv")

# Evaluation

In [30]:
print(y)
print(scores)

[0 0 0 ... 0 0 0]
[-6.970554 -6.970554 -6.970554 ... -6.970554 -6.970554 -6.970554]
